In [7]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
repository_url = "https://github.com/Lhenzo/InstaGeo-E2E-Geospatial-ML.git"

!git clone {repository_url}

Cloning into 'InstaGeo-E2E-Geospatial-ML'...
remote: Enumerating objects: 404, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 404 (delta 98), reused 88 (delta 75), pack-reused 269 (from 1)
Receiving objects: 100% (404/404), 1.43 MiB | 23.29 MiB/s, done.
Resolving deltas: 100% (225/225), done.


In [2]:
%%bash
cd InstaGeo-E2E-Geospatial-ML
git checkout geo-ai-hack
git pull
git status

Branch 'geo-ai-hack' set up to track remote branch 'geo-ai-hack' from 'origin'.
Already up to date.
On branch geo-ai-hack
Your branch is up to date with 'origin/geo-ai-hack'.

nothing to commit, working tree clean


Switched to a new branch 'geo-ai-hack'


In [3]:
%%bash
cd InstaGeo-E2E-Geospatial-ML
pip install -e .[all]

Obtaining file:///kaggle/working/InstaGeo-E2E-Geospatial-ML
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of h5pyd to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlxtend 0.23.3 requires scikit-learn>=1.3.1, but you have scikit-learn 1.2.2 which is incompatible.
pandas-gbq 0.25.0 requires google-api-core<3.0.0dev,>=2.10.2, but you have google-api-core 1.34.1 which is incompatible.
plotnine 0.14.4 requires matplotlib>=3.8.0, but you have matplotlib 3.7.5 which is incompatible.
rapids-dask-dependency 24.12.0 requires dask==2024.11.2, but you have dask 2024.12.1 which is incompatible.
rapids-dask-dependency 24.12.0 requires dask-expr==1.1.19, but you have dask-expr 1.1.21 which is incompatible.
rapids-dask-dependency 24.12.0 requires distributed==2024.11.2, but you have distributed 2024.12.1 which is incompatible.
tensorflow-decision-forests 1.10.0 requires tensorflow==2.17.0, but you have tensorflow 2.17.1 which is incompatible.


In [4]:
# Import necessary libraries
import os
import re
import shutil
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from pyproj import CRS, Transformer
import rasterio
os.environ["HYDRA_FULL_ERROR"] ="1"

In [5]:
def load_yml(filepath):
    """Load data from a YAML file.

    Args:
        filepath (str | Path): The path to the YAML file.

    Returns:
        Dict: The loaded data, or None if the file does not exist.
    """
    filepath=Path(filepath)
    with filepath.open() as f:
        return yaml.safe_load(f)

def save_yml(data,filepath):
    """Save data to a YAML file.

    Args:
        data (Dict): The data to save.
        filepath (str | Path): The file path to save the YAML to.
    """
    filepath = Path(filepath)
    with filepath.open("w") as f:
        yaml.dump(data, f)
    print(f"Data saved to {filepath}.")


## TO BE FILLED BY USER

In [14]:
# Updat locust file
# Load the Locust model configuration file
locust_cfg_path="InstaGeo-E2E-Geospatial-ML/instageo/model/configs/locust.yaml"
# Load the YAML configuration into a dictionary
locust_cfg=load_yml(locust_cfg_path)
# Update the mean and standard deviation values in the configuration
locust_cfg["mean"]=[670.5441284179688, 1267.7974853515625, 1772.599365234375, 2415.69091796875, 2879.2431640625, 2337.822509765625]
locust_cfg["std"]=[2146.305419921875, 2203.416259765625, 2247.03515625, 2310.74755859375, 2322.708984375, 2211.968505859375]
locust_cfg["model"]["freeze_backbone"] = True
locust_cfg["train"]["learning_rate"] = 0.001 
locust_cfg["dataloader"]["temporal_dim"] = 4

# Save the updated configuration back to the YAML file
locust_cfg_path_new = "InstaGeo-E2E-Geospatial-ML/instageo/model/configs/locust-hack.yaml"
save_yml(locust_cfg, locust_cfg_path_new)

Data saved to InstaGeo-E2E-Geospatial-ML/instageo/model/configs/locust-hack.yaml.


In [21]:
def generate_label_mapping(root_dir, input_subdir, output_csv):
    """
    Generate a CSV mapping input chips to corresponding segmentation maps.

    Args:
        root_dir (str or Path): Root directory containing the subdirectories for chips and segmentation maps.
        input_subdir (str): Subdirectory path for chips within the root directory.
        output_csv (str or Path): Output path for the generated CSV file.
    """
    root_dir = Path(root_dir)
    chips_orig = os.listdir(root_dir / input_subdir / "chips")
    if os.path.exists(root_dir / input_subdir / "seg_maps"):
        add_label = True
    else:
        add_label = False

    chips = [chip.replace("chip", f"{input_subdir}/chips/chip") for chip in chips_orig]

    if add_label:
        seg_maps = [chip.replace("chip", f"{input_subdir}/seg_maps/seg_map") for chip in chips_orig]
        df = pd.DataFrame({"Input": chips, "Label": seg_maps})
    else:
        df = pd.DataFrame({"Input": chips})
    df.to_csv(output_csv, index=False)

    print(f"Number of rows is: {df.shape[0]}")
    print(f"CSV generated and saved to: {output_csv}")

# TO BE FILLED: set data folder path
input_dir="/kaggle/input/geo-ai-hack"

# Generate label mappings for the training and testing datasets
generate_label_mapping(input_dir, 'hls_train/hls_train', "train_ds.csv")
generate_label_mapping(input_dir, 'hls_test/hls_test', "test_ds.csv")

def filter_dataset(in_path, out_path):
    train_ds = pd.read_csv(in_path)
    train_ds["Datetime"] = train_ds["Label"].apply(
        lambda row: pd.to_datetime(row.split("/")[-1].split("_")[2])
    )
    train_ds[train_ds["Datetime"].dt.year >= 2020].to_csv(out_path)

filter_dataset("train_ds.csv", "train_small_ds.csv")

def split_validation_data(mapping_csv, validation_split=0.3):
    """
    Split data into training and validation sets based on a CSV file mapping `chips` and `seg_maps`.

    Args:
        mapping_csv (str or Path): Path to the CSV file containing the mapping between `chips` and `seg_maps`.
        data_dir (str or Path): Path to the merged directory containing all files.
        validation_dir (str or Path): Path to the new directory for validation files.
        validation_split (float): Fraction of the data to use as the validation set.
    """
    df = pd.read_csv(mapping_csv)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    num_val = int(len(df) * validation_split)
    train_df = df[num_val:]
    val_df = df[:num_val]
    train_df.to_csv("train_split.csv",index=False)
    print(f"CSV train split  saved to: train_split.csv")
    val_df.to_csv("validation_split.csv",index=False)
    print(f"CSV validation split  saved to: validation_split.csv")

    return


# Split the training dataset into training and validation sets
split_validation_data(
    mapping_csv="train_ds.csv",
    validation_split=0.2
)

Number of rows is: 10428
CSV generated and saved to: train_ds.csv
Number of rows is: 2404
CSV generated and saved to: test_ds.csv
CSV train split  saved to: train_split.csv
CSV validation split  saved to: validation_split.csv


In [30]:
ls -a /k

./                           test_ds.csv         validation_split.csv
../                          train_ds.csv        .virtual_documents/
InstaGeo-E2E-Geospatial-ML/  train_small_ds.csv
outputs/                     train_split.csv


In [ ]:
!python -m instageo.model.run  --config-name=locust-hack \
hydra.run.dir="/kaggle/working/outputs/first_run" \
root_dir="/kaggle/input/geo-ai-hack" \
train.batch_size=8\
train.num_epochs=20 \
mode=train \
train_filepath="train_split.csv" \
valid_filepath="validation_split.csv"

[2025-02-04 20:18:42,949][__main__][INFO] - Script: /kaggle/working/InstaGeo-E2E-Geospatial-ML/instageo/model/run.py
[2025-02-04 20:18:42,953][__main__][INFO] - Imported hydra config:
checkpoint_path: null
dataloader:
  bands:
  - 0
  - 1
  - 2
  - 3
  - 4
  - 5
  - 6
  - 7
  - 8
  - 9
  - 10
  - 11
  - 12
  - 13
  - 14
  - 15
  - 16
  - 17
  constant_multiplier: 1.0
  img_size: 256
  mean:
  - 623.2724609375
  - 1247.657958984375
  - 1772.24169921875
  - 2371.256103515625
  - 2862.867431640625
  - 2357.759765625
  no_data_value: -9999
  reduce_to_zero: false
  replace_label:
  - -9999
  - -1
  std:
  - 2182.050048828125
  - 2248.420654296875
  - 2302.53515625
  - 2372.204345703125
  - 2398.52685546875
  - 2292.96435546875
  temporal_dim: 4
mean:
- 670.5441284179688
- 1267.7974853515625
- 1772.599365234375
- 2415.69091796875
- 2879.2431640625
- 2337.822509765625
mode: train
model:
  freeze_backbone: true
  num_classes: 2
output_dir: null
root_dir: /kaggle/input/geo-ai-hack
std:
- 2146.

In [ ]:
!python -m instageo.model.run --config-name=locust-hack \
    root_dir="/kaggle/input/geo-ai-hack" \
    test_filepath="validation_split.csv" \
    train.batch_size=8 \
    checkpoint_path='checkpoint-path' \
    mode=eval

In [27]:
from pyproj import CRS, Transformer
import os
import rasterio
import numpy as np

predictions_directory = "predictions"
prediction_files = os.listdir(predictions_directory)

def get_prediction_value(row):
    matching_files = [f for f in prediction_files if (str(row['date']) in f) and (row['mgrs_tile_id'] in f)]
    if not matching_files:
        return (np.nan, np.nan)
    for file in matching_files:
        with rasterio.open(f"{predictions_directory}/{file}") as src:
            width, height = src.width, src.height
            affine_transform = rasterio.transform.AffineTransformer(src.transform)
            transformer = Transformer.from_crs(CRS.from_epsg(4326), src.crs, always_xy=True)
            x_chip, y_chip = transformer.transform(row['x'], row['y'])
            x_offset, y_offset = affine_transform.rowcol(x_chip, y_chip)
            
            if 0 <= x_offset < width and 0 <= y_offset < height:
                return src.read(1)[x_offset, y_offset], file
    return (np.nan, np.nan)

FileNotFoundError: [Errno 2] No such file or directory: 'predictions'

In [ ]:
submission_df = pd.read_csv("hls_submission.csv")
submission_df[['prediction', 'filename']] = submission_df.apply(get_prediction_value, axis=1, result_type='expand')
submission_df.to_csv("hls_submission.csv")

In [29]:
! ls -a /kaggle/input/geo-ai-hack

.   geoai_hack.ipynb  hls_train		     starter-notebook_kaggle_V1.ipynb
..  hls_test	      sample_submission.csv  test.csv
